In [1]:
import pandas as pd
from pathlib import Path

root = Path.cwd()

if root.name == "notebooks":
    root = root.parent

data_path = root / "data" / "processed" / "abcd_first_msgs.parquet"

df = pd.read_parquet(data_path)
df.head()

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me ...",0,1,13
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,0,10
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to kn...,0,0,27
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deat...",0,1,18
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,0,0,34


In [2]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS) - {
    "no", "not", "nor", "never", "back", "get", "give"
}

def clean_message(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().replace("’", "'").strip()
    text = text.replace("won't", "will not")
    text = text.replace("can't", "can not")
    text = text.replace("n't", " not")

    text = re.sub(r"\bho!", "", text)
    text = re.sub(
        r"^(hi+|hello+|helo+|hey+|hiya|good morning|good afternoon|good evening)\b[\s,.!]*",
        "",
        text
    )
    text = re.sub(r"[^\w\s]|_", " ", text)

    return " ".join(word for word in text.split() if word not in stop_words)

df["clean_text"] = df["first_msgs"].apply(clean_message)

df[["first_msgs", "clean_text"]].head(10)

,first_msgs,clean_text
0,"Hi! I need to return an item, can you help me ...",need return item help
1,just wanted to check on the status of a refund,just wanted check status refund
2,HEY HO! I've got a promo code and I want to kn...,ve got promo code want know expire d like use ...
3,"Hello, I would like to change my shipping deat...",like change shipping deatails changed recently
4,Hello. I have a really cool party coming up. A...,really cool party coming need new clothes asap...
5,I want an extension for my shopping service. I...,want extension shopping service need time get ...
6,"Hello, I am trying to check the status of my o...",trying check status order forgot username not ...
7,Hello I am trying to buy a pair of boots but t...,trying buy pair boots not getting added cart
8,I accidently ordered the wrong item and was wo...,accidently ordered wrong item wondering change...
9,I would like to confirm my address is correct ...,like confirm address correct order able


In [3]:
output_path = root / "data" / "processed" / "abcd_cleaned.parquet"

df.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Saved: /Users/oscaralawi/code/MUlbing/ClaimLens-review/data/processed/abcd_cleaned.parquet
Rows: 10042
Columns: ['split', 'id', 'flow', 'subflow', 'first_msgs', 'refund', 'refund_label', 'n_words', 'clean_text']
